# 検索しやすい形に質問を書き換える（文脈の補完と HyDE）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/23_rag/rag_query_rewrite.ipynb)

NLP クックブック 23-05 ／ RAG（検索拡張生成）

対話の途中の「それはいくら？」のような質問は、そのままでは検索に当たらない。質問を検索向けに書き換えたい。

In [1]:
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

docs = [
    "クックブック食堂の営業時間は平日 11 時から 15 時、土日は 11 時から 20 時です。月曜日は定休日です。",
    "クックブック食堂の看板メニューは、3 日間煮込んだ特製ビーフカレー (950 円) です。大盛りは 100 円増しです。",
    "クックブック食堂は 2019 年に京都市左京区で開業しました。店主の田中さんはインドで 2 年間修行しました。",
    "駐車場は店舗の裏に 3 台分あります。満車の場合は近隣のコインパーキングをご利用ください。",
    "予約は電話 (075-000-0000) のみ受け付けています。5 名以上の場合は前日までにご連絡ください。",
    "支払いは現金と QR コード決済に対応しています。クレジットカードは使えません。",
]
embedder = SentenceTransformer("intfloat/multilingual-e5-small")
doc_emb = embedder.encode(["passage: " + d for d in docs], normalize_embeddings=True)

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

def chat(messages, max_new_tokens=80):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def top(query, k=2, prefix="query: "):
    sims = doc_emb @ embedder.encode(prefix + query, normalize_embeddings=True)
    return [(round(float(sims[i]), 3), docs[i][:32]) for i in sims.argsort()[::-1][:k]]

In [2]:
# 1. 対話の文脈を補って、単独で意味が通じる質問にする
def rewrite(history, question):
    dialogue = "\n".join(f"{'ユーザー' if role == 'user' else 'アシスタント'}: {text}" for role, text in history)
    prompt = ("次の対話の最後の質問を、対話を読んでいない人にも意味が通じる、検索用の 1 つの質問に書き換えてください。"
              f"書き換えた質問だけを出力してください。\n\n{dialogue}\nユーザー: {question}")
    return chat([{"role": "user", "content": prompt}]).strip("「」")

history = [("user", "クックブック食堂の看板メニューは何ですか？"), ("assistant", "3 日間煮込んだ特製ビーフカレーです。")]
for q in ["それはいくらですか？", "大盛りにできますか？", "車で行っても大丈夫？"]:
    r = rewrite(history, q)
    print(f"{q} → {r}\n   そのまま検索: {top(q, 1)}\n   書き換えて検索: {top(r, 1)}")

それはいくらですか？ → クックブック食堂の看板メニューであるビーフカレーの値段はいくらですか？
   そのまま検索: [(0.817, '支払いは現金と QR コード決済に対応しています。クレジットカー')]
   書き換えて検索: [(0.927, 'クックブック食堂の看板メニューは、3 日間煮込んだ特製ビーフカレ')]
大盛りにできますか？ → クックブック食堂の看板メニューであるビーフカレーを大盛りにしてもらうことは可能か？
   そのまま検索: [(0.813, 'クックブック食堂の看板メニューは、3 日間煮込んだ特製ビーフカレ')]
   書き換えて検索: [(0.912, 'クックブック食堂の看板メニューは、3 日間煮込んだ特製ビーフカレ')]
車で行っても大丈夫？ → クックブック食堂の看板メニューである特製ビーフカレーは、車でのアクセスは可能ですか？
   そのまま検索: [(0.817, '駐車場は店舗の裏に 3 台分あります。満車の場合は近隣のコインパ')]
   書き換えて検索: [(0.89, 'クックブック食堂の看板メニューは、3 日間煮込んだ特製ビーフカレ')]


In [3]:
# 2. HyDE: 質問に対する「仮の答え」を LLM に書かせ、その文章で検索する (質問文より、答えの文のほうが文書に似ている)
def hyde(question):
    prompt = f"次の質問に対する答えとして、ありそうな内容を飲食店の案内文の形で 1 文だけ書いてください。事実でなくて構いません。\n\n質問: {question}"
    return chat([{"role": "user", "content": prompt}])

for q in ["カードで払いたいんだけど", "大人数で行きたい"]:
    h = hyde(q)
    print(f"{q}\n   仮の答え: {h}\n   そのまま検索: {top(q, 1)}\n   HyDE で検索: {top(h, 1, prefix='passage: ')}")

カードで払いたいんだけど
   仮の答え: 当レストランでは、mastercardやvisaはもちろん、american expressやdiners clubもご利用頂け、さらにapple payやqrコード決済にも対応しています。カードでのお支払いが便利にご利用頂けます。
   そのまま検索: [(0.864, '支払いは現金と QR コード決済に対応しています。クレジットカー')]
   HyDE で検索: [(0.894, '支払いは現金と QR コード決済に対応しています。クレジットカー')]
大人数で行きたい
   仮の答え: 「当店では、広々とした宴会場をご用意しており、最大50名様までご利用いただけるため、大人数でのお集まりに最適です。」
   そのまま検索: [(0.828, '予約は電話 (075-000-0000) のみ受け付けています。')]
   HyDE で検索: [(0.883, '予約は電話 (075-000-0000) のみ受け付けています。')]


- 複数ターンの RAG では、質問の書き換えはほぼ必須。「それ」「そこ」や省略された主語を補わないと、検索が的外れになる。
- HyDE (Hypothetical Document Embeddings) の仮の答えは内容が間違っていてよい。語彙や文体が文書に近づくことで検索が当たりやすくなる。ただし LLM の呼び出しが 1 回増える。
- 書き換えは失敗もする。3 つ目の例では、前の話題（ビーフカレー）を無関係な質問に持ち込んでしまい、書き換えないほうが正しい資料に当たっている。指示語や省略がある質問だけを書き換える、元の質問と書き換えた質問の両方で検索する、といった対策がある。
- 書き換えの結果は必ずログに残す。回答がおかしいとき、書き換えで意味が変わっていた、ということがよくある。